# 19. Model Training for Late Delivery Prediction

In the previous notebook, the raw Olist datasets were cleaned, aggregated, merged, and transformed into an order-level modeling dataset. The final processed training and testing files were saved so that the preprocessing work would not need to be repeated.

In this notebook, the processed data is loaded and used to train machine learning models that predict whether an order will be delivered late. The goal is to compare a baseline linear model and a more flexible tree-based model, then evaluate their predictive performance using classification metrics.

# 20. Import Required Libraries

This section imports the libraries needed for loading the processed datasets, training classification models, and evaluating model performance.

The notebook uses scikit-learn to train and compare two models:

- Logistic Regression as a baseline model
- Random Forest as a stronger non-linear model

In [1]:
import os
import boto3
import pandas as pd
import numpy as np
from io import BytesIO

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.9 
Trying to create a Glue session for the kernel.
Session Type: etl
Session ID: 674efa80-755a-478d-b6fa-0c3df65d8f87
Applying the following default arguments:
--glue_kernel_version 1.0.9
--enable-glue-datacatalog true
Waiting for session 674efa80-755a-478d-b6fa-0c3df65d8f87 to get into ready status...
Session 674efa80-755a-478d-b6fa-0c3df65d8f87 has been created.



# 21. Load the Processed Training and Testing Datasets

To ensure that preprocessing work is preserved and reusable, the processed training and testing files are loaded directly from Amazon S3. This avoids rebuilding the preprocessing pipeline and ensures that the exact same prepared dataset is used for model training.

The training and testing files were saved in the earlier preprocessing notebook with the target variable placed in the first column.

In [2]:
bucket_name = "cartwave"

s3 = boto3.client("s3")

def read_csv_from_s3(bucket, key, header=None):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(BytesIO(obj["Body"].read()), header=header)

train_df = read_csv_from_s3(bucket_name, "processed/train.csv", header=None)
test_df = read_csv_from_s3(bucket_name, "processed/test.csv", header=None)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTrain preview:")
print(train_df.head())
print("\nTest preview:")
print(test_df.head())

Train shape: (77176, 17)
Test shape: (19294, 17)

Train preview:
   0   1       2      3       4   5   ...  11  12  13  14        15         16
0   0   1  189.90  17.60  189.90   1  ...  25   7   1  22  0.321667   8.064850
1   0   1   34.99   8.64   34.99   1  ...  25   3   4  14  0.254722  11.388831
2   0   1   79.90  23.49   79.90   1  ...  22   4   4  10  0.826389  20.561319
3   0   1   32.90  17.60   32.90   1  ...  17   6   3  19  0.371944  21.202778
4   0   1   69.90  17.77   69.90   1  ...  26   5   5   0  1.051667  30.986331

[5 rows x 17 columns]

Test preview:
   0   1      2      3      4   5   ...  11  12  13  14          15         16
0   0   1  325.0  23.07  325.0   1  ...   9   2   5  20  131.255000  25.138495
1   0   1  129.9  15.66  129.9   1  ...  25   1   1  16    0.303056  23.302685
2   0   2  276.0  26.62  138.0   1  ...  25   6   2  15    0.218056  20.349201
3   0   1   28.9  14.52   28.9   1  ...  18   4   3  13    0.248611  21.453414
4   0   1  104.9  43.14  104

# 22. Separate Features and Target Variable

The processed training and testing files were saved with the target variable in the first column. In this section, the target is separated from the predictor variables so the data can be passed into machine learning models correctly.

The first column represents the `late_delivery_flag`, where:

- 1 indicates a late delivery
- 0 indicates an on-time or early delivery

In [3]:
y_train = train_df.iloc[:, 0]
X_train = train_df.iloc[:, 1:]

y_test = test_df.iloc[:, 0]
X_test = test_df.iloc[:, 1:]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (77176, 16)
X_test shape: (19294, 16)
y_train shape: (77176,)
y_test shape: (19294,)

Training target distribution:
0    0.918874
1    0.081126
Name: 0, dtype: float64

Testing target distribution:
0    0.918887
1    0.081113
Name: 0, dtype: float64


# 23. Train a Baseline Logistic Regression Model

Logistic Regression is used as the first model because it is simple, interpretable, and commonly used as a baseline for binary classification problems.

This model provides a useful performance reference point. Although it assumes a linear relationship between the predictors and the log-odds of the target, it often performs well enough to establish whether the classification problem is learnable.

In [4]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

log_reg_pred = log_reg.predict(X_test)
log_reg_prob = log_reg.predict_proba(X_test)[:, 1]

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.
/home/spark/.local/lib/python3.7/site-packages/sklearn/linear_model/_logistic.py:765: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG)


# 24. Evaluate Logistic Regression Performance

After training the baseline model, its performance is evaluated on the testing dataset. Several classification metrics are used, including accuracy, precision, recall, F1-score, and ROC-AUC.

These metrics provide a balanced view of model quality and help assess how well the model identifies delayed deliveries.

In [5]:
log_reg_accuracy = accuracy_score(y_test, log_reg_pred)
log_reg_precision = precision_score(y_test, log_reg_pred)
log_reg_recall = recall_score(y_test, log_reg_pred)
log_reg_f1 = f1_score(y_test, log_reg_pred)
log_reg_auc = roc_auc_score(y_test, log_reg_prob)

print("Logistic Regression Performance:")
print("Accuracy :", round(log_reg_accuracy, 4))
print("Precision:", round(log_reg_precision, 4))
print("Recall   :", round(log_reg_recall, 4))
print("F1 Score :", round(log_reg_f1, 4))
print("ROC-AUC  :", round(log_reg_auc, 4))

print("\nClassification Report:")
print(classification_report(y_test, log_reg_pred))

Logistic Regression Performance:
Accuracy : 0.9189
Precision: 0.5
Recall   : 0.0006
F1 Score : 0.0013
ROC-AUC  : 0.6236

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     17729
           1       0.50      0.00      0.00      1565

    accuracy                           0.92     19294
   macro avg       0.71      0.50      0.48     19294
weighted avg       0.88      0.92      0.88     19294


# 25. Analyze Logistic Regression with a Confusion Matrix

A confusion matrix helps interpret the model's prediction behavior by showing the number of correct and incorrect classifications.

This is especially useful for understanding whether the model is better at identifying on-time deliveries or delayed deliveries.

In [6]:
log_reg_cm = confusion_matrix(y_test, log_reg_pred)
print("Confusion Matrix for Logistic Regression:")
print(log_reg_cm)

plt.figure(figsize=(5, 4))
plt.imshow(log_reg_cm, interpolation='nearest')
plt.title("Logistic Regression Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ["Pred 0", "Pred 1"])
plt.yticks([0, 1], ["Actual 0", "Actual 1"])

for i in range(log_reg_cm.shape[0]):
    for j in range(log_reg_cm.shape[1]):
        plt.text(j, i, log_reg_cm[i, j], ha="center", va="center")

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

Confusion Matrix for Logistic Regression:
[[17728     1]
 [ 1564     1]]


# 26. Train a Random Forest Model

Random Forest is used as a second model because it can capture more complex, non-linear relationships in the data. It is also robust to many feature interactions and often performs better than linear models on structured tabular datasets.

This model will be compared against Logistic Regression to determine whether the additional flexibility improves late delivery prediction.

In [7]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


# 27. Evaluate Random Forest Performance

The Random Forest model is evaluated using the same classification metrics as the baseline model. This allows a direct comparison between the two approaches.

If the Random Forest model achieves stronger recall, F1-score, or ROC-AUC, it may be a better choice for the business problem.

In [8]:
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest Performance:")
print("Accuracy :", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall   :", round(rf_recall, 4))
print("F1 Score :", round(rf_f1, 4))
print("ROC-AUC  :", round(rf_auc, 4))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

Random Forest Performance:
Accuracy : 0.9194
Precision: 0.7083
Recall   : 0.0109
F1 Score : 0.0214
ROC-AUC  : 0.7589

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     17729
           1       0.71      0.01      0.02      1565

    accuracy                           0.92     19294
   macro avg       0.81      0.51      0.49     19294
weighted avg       0.90      0.92      0.88     19294


# 28. Analyze Random Forest with a Confusion Matrix

A confusion matrix is also generated for the Random Forest model so that its classification behavior can be compared with the baseline model.

This helps determine whether the Random Forest improves identification of late deliveries without causing too many false alarms.

In [9]:
rf_cm = confusion_matrix(y_test, rf_pred)
print("Confusion Matrix for Random Forest:")
print(rf_cm)

plt.figure(figsize=(5, 4))
plt.imshow(rf_cm, interpolation='nearest')
plt.title("Random Forest Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ["Pred 0", "Pred 1"])
plt.yticks([0, 1], ["Actual 0", "Actual 1"])

for i in range(rf_cm.shape[0]):
    for j in range(rf_cm.shape[1]):
        plt.text(j, i, rf_cm[i, j], ha="center", va="center")

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

Confusion Matrix for Random Forest:
[[17722     7]
 [ 1548    17]]


# 29. Compare Model Performance

To make the results easier to interpret, the evaluation metrics for both models are summarized side by side. This helps identify which model performs better overall and which model may be more suitable for the business objective.

Because delayed delivery prediction is a business risk problem, metrics such as recall and F1-score are often especially important.

In [10]:
results_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [log_reg_accuracy, rf_accuracy],
    "Precision": [log_reg_precision, rf_precision],
    "Recall": [log_reg_recall, rf_recall],
    "F1 Score": [log_reg_f1, rf_f1],
    "ROC-AUC": [log_reg_auc, rf_auc]
})

print(results_df)

                 Model  Accuracy  Precision    Recall  F1 Score   ROC-AUC
0  Logistic Regression  0.918887   0.500000  0.000639  0.001276  0.623584
1        Random Forest  0.919405   0.708333  0.010863  0.021397  0.758932


# 30. Examine Feature Importance

Random Forest provides feature importance scores that indicate which predictors contributed most to the model's decisions.

These scores do not prove causality, but they are useful for understanding which operational or transactional factors are most associated with late deliveries.

In [11]:
feature_names = [
    "num_items",
    "total_price",
    "total_freight",
    "avg_item_price",
    "num_unique_sellers",
    "num_payment_records",
    "total_payment_value",
    "max_installments",
    "main_payment_type",
    "customer_city",
    "customer_state",
    "purchase_month",
    "purchase_dayofweek",
    "purchase_hour",
    "approval_delay_hours",
    "estimated_delivery_days"
]

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(feature_importance_df)

                    Feature  Importance
15  estimated_delivery_days    0.226769
11           purchase_month    0.187811
10           customer_state    0.100609
14     approval_delay_hours    0.092349
2             total_freight    0.084069
9             customer_city    0.058413
6       total_payment_value    0.054985
3            avg_item_price    0.050011
1               total_price    0.049643
13            purchase_hour    0.031894
12       purchase_dayofweek    0.021883
7          max_installments    0.017826
8         main_payment_type    0.009779
0                 num_items    0.006809
5       num_payment_records    0.005100
4        num_unique_sellers    0.002051


# 31. Visualize Feature Importance

This chart provides a visual summary of the most influential features in the Random Forest model. It helps translate technical model output into interpretable business insight.

In [20]:
import matplotlib.pyplot as plt

top_features = feature_importance_df.head(10).sort_values("Importance", ascending=True)

print("Top 10 Feature Importances:")
print(top_features)

plt.figure(figsize=(8, 5))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top 10 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

Top 10 Feature Importances:
                    Feature  Importance
13            purchase_hour    0.031894
1               total_price    0.049643
3            avg_item_price    0.050011
6       total_payment_value    0.054985
9             customer_city    0.058413
2             total_freight    0.084069
14     approval_delay_hours    0.092349
10           customer_state    0.100609
11           purchase_month    0.187811
15  estimated_delivery_days    0.226769


In [21]:
top_features = feature_importance_df.head(10).sort_values("Importance", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top 10 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# 32. Select the Best Model

Based on the evaluation results, the better-performing model can be selected as the final recommendation for the first version of the project.

The selected model should balance predictive accuracy with business usefulness. In this project, identifying delayed deliveries early is valuable because it can support proactive interventions, customer communication, and seller monitoring.

In [22]:
best_model_name = results_df.sort_values(by="F1 Score", ascending=False).iloc[0]["Model"]
print("Recommended model based on F1 Score:", best_model_name)

Recommended model based on F1 Score: Random Forest


# 33. Upload Model Results to Amazon S3

To ensure the model evaluation outputs are not lost after the notebook session ends, the results file is uploaded to Amazon S3. This makes the output reusable for reporting, presentation preparation, and future project iterations.

In [27]:
from io import StringIO
import boto3

bucket_name = "cartwave"
s3 = boto3.client("s3")

csv_buffer = StringIO()
results_df.to_csv(csv_buffer, index=False)

s3.put_object(
    Bucket=bucket_name,
    Key="processed/model_results.csv",
    Body=csv_buffer.getvalue()
)

print("Uploaded to s3://cartwave/processed/model_results.csv")

Uploaded to s3://cartwave/processed/model_results.csv


# 34. Modeling Summary

In this notebook, the processed training and testing datasets were loaded from Amazon S3 and used to train two classification models: Logistic Regression and Random Forest.

The models were evaluated using accuracy, precision, recall, F1-score, ROC-AUC, classification reports, and confusion matrices. The comparison showed which modeling approach was more effective for predicting delayed deliveries. Feature importance from the Random Forest model also provided insight into which variables were most strongly associated with late deliveries.

These results can now be used to support the business recommendation and executive presentation for CartWave Marketplace.